# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema, available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print("Description:")
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We use the dataset's record set structure to enumerate available tables and their schemas.

In [ ]:
# List all record sets in the dataset, with their @id and available fields.
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    record_sets = []

if not record_sets:
    # Try to infer record sets from the dataset if they are not present in metadata
    print("No record sets declared explicitly in metadata; attempting to enumerate automatically...")
    # mlcroissant auto-discovers record sets via dataset._record_sets (private, but used as fallback)
    record_sets = [rs['@id'] for rs in dataset._record_sets]
    print("Record set @id list:")
    for idx, record_set_id in enumerate(record_sets):
        print(f"  [{idx}]  {record_set_id}")
else:
    print("Record Sets found in metadata:")
    for idx, rs in enumerate(record_sets):
        # rs may be an object or dict with an '@id'
        if isinstance(rs, dict) and '@id' in rs:
            print(f"  [{idx}]  {rs['@id']}")
        elif hasattr(rs, '@id'):
            print(f"  [{idx}]  {rs.@id}")
        else:
            print(f"  [{idx}]  {rs}")

# Optionally preview fields in the first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"\nFields for record set: {first_rs_id}")
    for rec in dataset.records(record_set=first_rs_id):
        print("Record example:")
        pprint.pprint(rec)
        break  # show only the first record


## 3. Data Extraction
Load data from each record set (by its `@id`) into DataFrames for further exploration. Here, we use the discovered record set `@id`s.

In [ ]:
# Extract records for each available record set
dataframes = {}
# Ensure record_sets is a list of @ids
record_set_ids = record_sets

for record_set_id in record_set_ids:
    # Use mlcroissant's iterator over records
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f"No records found for record set: {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Typical EDA operations include filtering, normalization, and aggregation/grouping. We'll select a numeric field (column) from one record set (by its `@id`) to demonstrate these steps. You may adjust the variables in the code for the specific record set and field you wish to explore.

In [ ]:
# --- EDA Placeholder: edit as appropriate for the dataset ---
# For demonstration, we use the first available record set and try to auto-select a numeric field
import numpy as np

if dataframes:
    selected_record_set_id = next((k for k, v in dataframes.items() if not v.empty), None)
    if selected_record_set_id is not None:
        df = dataframes[selected_record_set_id].copy()
        
        # Try to find a numeric field
        numeric_field = None
        for col in df.columns:
            try:
                # Try casting to numeric, ignore errors
                vals = pd.to_numeric(df[col], errors='coerce')
                if vals.notnull().sum() > 0 and vals.dtype in ["float64", "int64"]:
                    numeric_field = col
                    break
            except Exception:
                continue
        
        if numeric_field:
            print(f"Using numeric field: {numeric_field} from record set: {selected_record_set_id}")
            
            threshold = df[numeric_field].dropna().median()  # Use median as threshold
            print(f"Using threshold = median = {threshold}")
            
            filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())

            # Normalize the numeric field (z-score)
            filtered_df = filtered_df.copy()
            filtered_df["{}_normalized".format(numeric_field)] = (
                pd.to_numeric(filtered_df[numeric_field], errors="coerce") - pd.to_numeric(filtered_df[numeric_field], errors="coerce").mean()
            ) / pd.to_numeric(filtered_df[numeric_field], errors="coerce").std()
            print(f"\nNormalized {numeric_field}:")
            print(filtered_df[[numeric_field, '{}_normalized'.format(numeric_field)]].head())

            # Try grouping by a categorical column if available
            group_field = None
            for col in df.columns:
                if col != numeric_field and df[col].nunique() < 20:
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nGrouped data by {group_field} and mean of {numeric_field}:")
                print(grouped_df.head())
            else:
                print("No appropriate group field found for grouping.")
        else:
            print("No numeric field found in the first non-empty record set.")
    else:
        print("No non-empty record set found.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationships with other fields.

In [ ]:
import matplotlib.pyplot as plt

# Visualize numeric field, if available
if 'filtered_df' in locals() and numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    values = pd.to_numeric(filtered_df[numeric_field], errors="coerce").dropna()
    plt.hist(values, bins=20, alpha=0.7)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field} (filtered)')
    plt.show()
    # If grouping available, plot group means
    if 'group_field' in locals() and group_field and 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field])
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field loaded or available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and perform initial EDA on a dataset described by a Croissant schema using `mlcroissant`.

- We loaded the dataset and inspected its metadata and available record sets, referencing all items via their `@id`.
- We extracted records into DataFrames for analysis.
- We performed basic EDA including filtering, normalization, and group aggregation on available fields.
- Visualization allowed insights into quantitative features.

Continue with advanced analysis as needed using the DataFrames loaded from this robust, richly annotated FAIR² resource.